# Online Retail Data Cleaning

This notebook imports the `Online Retail.xlsx` dataset and performs initial exploratory data analysis to identify data quality issues, anomalies, missing values, and structural problems.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
%matplotlib inline

## 1. Load the Dataset

In [2]:
# Load the Excel file
raw_data_path = Path("../data/raw/Online Retail.xlsx")
df = pd.read_excel(raw_data_path)

# Display basic info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [3]:
# View first few rows
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


## 2. Initial Data Inspection & Problem Identification

In [4]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage (%)": missing_percentage
})
print(missing_df)

             Missing Count  Missing Percentage (%)
InvoiceNo                0                0.000000
StockCode                0                0.000000
Description           1454                0.268311
Quantity                 0                0.000000
InvoiceDate              0                0.000000
UnitPrice                0                0.000000
CustomerID          135080               24.926694
Country                  0                0.000000


In [5]:
# Statistical summary of numerical columns
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [6]:
# Check statistical summary of non-numerical columns
df.describe(include=["object", "str"])

,InvoiceNo,StockCode,Description,Country
count,541909,541909,540455,541909
unique,25900,4070,4223,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
freq,1114,2313,2369,495478


### Checking Specific Data Quality Issues:
1. **Cancellations / Returns**: Invoice numbers starting with 'C'.
2. **Negative Quantities**: Quantities <= 0.
3. **Zero or Negative Unit Prices**: UnitPrice <= 0.
4. **Non-standard StockCodes**: Administrative or postal charges (e.g., 'POST', 'DOT', 'M', 'BANK CHARGES').

In [7]:
# 1. Check InvoiceNo prefixes
print("InvoiceNo prefix counts:")
print(df["InvoiceNo"].astype(str).str[0].value_counts())

# 2. Negative quantities
neg_qty = (df["Quantity"] < 0).sum()
print("Rows with negative Quantity: " + str(neg_qty))

# 3. Zero or negative UnitPrice
zero_price = (df["UnitPrice"] <= 0).sum()
print("Rows with UnitPrice <= 0: " + str(zero_price))

# 4. Check unique countries
print("Number of unique countries: " + str(df["Country"].nunique()))
print(df["Country"].value_counts().head(10))

InvoiceNo prefix counts:
InvoiceNo
5    532618
C      9288
A         3
Name: count, dtype: int64
Rows with negative Quantity: 10624
Rows with UnitPrice <= 0: 2517
Number of unique countries: 38
Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64


In [8]:
# Check non-standard StockCodes (non-numeric or special characters)
non_standard_stock = df[~df["StockCode"].astype(str).str.contains("^[0-9]", regex=True, na=False)]
print("Count of non-standard StockCodes: " + str(len(non_standard_stock)))
print(non_standard_stock["StockCode"].value_counts().head(15))

Count of non-standard StockCodes: 2995
StockCode
POST            1256
DOT              710
M                571
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
gift_0001_20      10
gift_0001_10       9
gift_0001_30       8
DCGS0003           5
Name: count, dtype: int64


---
# Step 1: Removing Duplicate Records

Systematic logging errors can result in completely identical rows being recorded multiple times. We will identify and remove duplicate rows, followed by a verification step.

In [9]:
# Check initial row count
initial_count = len(df)
print("Initial row count: " + format(initial_count, ","))

# Identify number of duplicate rows
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows: " + format(duplicate_count, ","))

# Drop duplicate rows
df = df.drop_duplicates()

# Verification Step 1
final_count = len(df)
print("Row count after removing duplicates: " + format(final_count, ","))
print("Rows successfully removed: " + format(initial_count - final_count, ","))
print("Remaining duplicate count: " + str(df.duplicated().sum()))

Initial row count: 541,909
Number of duplicate rows: 5,268
Row count after removing duplicates: 536,641
Rows successfully removed: 5,268
Remaining duplicate count: 0


---
# Step 2: Handling Missing Values

1. **CustomerID**: Fill missing values with `-1` (representing Guest Customers) and convert dtype from `float` to pandas nullable integer `Int64`.
2. **Description**: Drop rows where `Description` is missing.

In [10]:
# Check missing values before Step 2
print("Missing values before Step 2:")
print(df[["CustomerID", "Description"]].isnull().sum())

# 1. Handle CustomerID: Fill nulls with -1 and convert to Int64
df["CustomerID"] = df["CustomerID"].fillna(-1).astype("Int64")

# 2. Handle Description: Drop rows with missing Description
initial_len = len(df)
df = df.dropna(subset=["Description"])

# Verification Step 2
print("\n--- Verification Step 2 ---")
print("Rows dropped due to missing Description: " + format(initial_len - len(df), ","))
print("Missing values after Step 2:")
print(df[["CustomerID", "Description"]].isnull().sum())
print("CustomerID dtype: " + str(df["CustomerID"].dtype))
print("Guest Customer count (CustomerID == -1): " + format(int((df["CustomerID"] == -1).sum()), ","))

Missing values before Step 2:
CustomerID     135037
Description      1454
dtype: int64

--- Verification Step 2 ---
Rows dropped due to missing Description: 1,454
Missing values after Step 2:
CustomerID     0
Description    0
dtype: int64
CustomerID dtype: Int64
Guest Customer count (CustomerID == -1): 133,583


---
# Step 3: Handling Cancellations and Returns

Cancellations and returns are indicated by invoice numbers starting with 'C' or negative values in the Quantity column. Following commercial best practices, we will isolate these records into a dedicated `df_returns` DataFrame for return analysis, and retain only genuine purchases in our main `df` DataFrame.

In [11]:
# Identify cancellation and return rows
# Condition 1: InvoiceNo starts with 'C'
# Condition 2: Quantity < 0
cancellation_mask = (df["InvoiceNo"].astype(str).str.startswith("C")) | (df["Quantity"] < 0)

print(f"Total cancellation/return records identified: {cancellation_mask.sum():,}")

# Separate into dedicated df_returns DataFrame
df_returns = df[cancellation_mask].copy()

# Keep only non-cancelled transactions in main df
df = df[~cancellation_mask].copy()

# Verification Step 3
print("\n--- Verification Step 3 ---")
print(f"Main df row count after removing returns: {len(df):,}")
print(f"Dedicated df_returns row count: {len(df_returns):,}")
print(f"Remaining negative quantities in main df: {(df["Quantity"] < 0).sum()}")
print(f"Remaining 'C' invoices in main df: {df["InvoiceNo"].astype(str).str.startswith("C").sum()}")

Total cancellation/return records identified: 9,725

--- Verification Step 3 ---
Main df row count after removing returns: 525,462
Dedicated df_returns row count: 9,725
Remaining negative quantities in main df: 0
Remaining 'C' invoices in main df: 0


---
# Step 4: Handling Invalid Values and Outliers (`UnitPrice <= 0`)

Rows with zero or negative unit prices represent accounting errors, administrative fees, or adjustments rather than standard product purchases. We will inspect samples of these anomalous records and filter them out from our main revenue dataset.

In [12]:
# Identify rows with UnitPrice <= 0
zero_price_mask = df["UnitPrice"] <= 0
print(f"Total rows with UnitPrice <= 0: {zero_price_mask.sum():,}")

# Display sample records of invalid UnitPrice
print("\n--- Sample Records with UnitPrice <= 0 ---")
display(df[zero_price_mask].head(10))

# Filter out rows with UnitPrice <= 0
initial_len = len(df)
df = df[~zero_price_mask].copy()

# Verification Step 4
print("\n--- Verification Step 4 ---")
print(f"Rows removed due to UnitPrice <= 0: {initial_len - len(df):,}")
print(f"Main df row count after Step 4: {len(df):,}")
print(f"Remaining UnitPrice <= 0 count: {(df["UnitPrice"] <= 0).sum()}")

Total rows with UnitPrice <= 0: 584

--- Sample Records with UnitPrice <= 0 ---


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
6391,536941,22734,amazon,20,2010-12-03 12:08:00,0.0,-1,United Kingdom
6392,536942,22139,amazon,15,2010-12-03 12:08:00,0.0,-1,United Kingdom
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647,Germany
14335,537534,85064,CREAM SWEETHEART LETTER RACK,1,2010-12-07 11:48:00,0.0,-1,United Kingdom
14336,537534,84832,ZINC WILLIE WINKIE CANDLE STICK,1,2010-12-07 11:48:00,0.0,-1,United Kingdom
14337,537534,84692,BOX OF 24 COCKTAIL PARASOLS,2,2010-12-07 11:48:00,0.0,-1,United Kingdom
14338,537534,48184,DOORMAT ENGLISH ROSE,3,2010-12-07 11:48:00,0.0,-1,United Kingdom
14339,537534,48111,DOORMAT 3 SMILEY CATS,1,2010-12-07 11:48:00,0.0,-1,United Kingdom
14340,537534,22697,GREEN REGENCY TEACUP AND SAUCER,1,2010-12-07 11:48:00,0.0,-1,United Kingdom
14341,537534,22682,FRENCH BLUE METAL DOOR SIGN 7,1,2010-12-07 11:48:00,0.0,-1,United Kingdom



--- Verification Step 4 ---
Rows removed due to UnitPrice <= 0: 584
Main df row count after Step 4: 524,878
Remaining UnitPrice <= 0 count: 0


---
# Step 5: Keeping Only Genuine Product Codes

Genuine products in this dataset have a `StockCode` that starts with five digits
(optionally followed by a letter, e.g. `85123A`) or the DCGS product line (e.g. DCGS0069). Everything else is a non-product
record: postage (`POST`, `DOT`), manual entries (`M`), bank charges, carriage (`C2`),
gift vouchers, and similar. Instead of maintaining a hand-written blocklist, I keep only
the codes that match the product pattern. I also standardize `EIRE` to `Ireland`.

In [13]:
# 1. Inspect the most frequent non-product stock codes before filtering
is_standard_product = df["StockCode"].astype(str).str.match(r"^(?:\d{5}|DCGS)")
print("Most frequent non-product StockCodes (will be removed):")
display(df.loc[~is_standard_product, "StockCode"].value_counts().head(15))

# 2. Record initial row count
initial_len = len(df)

# 3. Keep only genuine product codes (starting with 5 digits)
is_standard_product = df["StockCode"].astype(str).str.match(r"^(?:\d{5}|DCGS)")
df = df[is_standard_product].copy()

# 4. Standardize country name: Rename 'EIRE' to 'Ireland'
df["Country"] = df["Country"].replace({"EIRE": "Ireland"})

# Verification Step 5
print("\n--- Verification Step 5 ---")
print(f"Total rows removed in Step 5: {initial_len - len(df):,}")
print(f"Main df row count after Step 5: {len(df):,}")
print(f"'Ireland' record count: {(df["Country"] == "Ireland").sum():,}")

Most frequent non-product StockCodes (will be removed):


StockCode
POST            1126
DOT              706
M                316
C2               141
BANK CHARGES      12
gift_0001_20       9
gift_0001_10       8
gift_0001_30       7
gift_0001_50       4
gift_0001_40       3
PADS               3
AMAZONFEE          2
S                  2
m                  1
B                  1
Name: count, dtype: int64


--- Verification Step 5 ---
Total rows removed in Step 5: 2,341
Main df row count after Step 5: 522,537
'Ireland' record count: 7,768


---
# Step 6: Initial Feature Engineering and Export

We will create key analytical features (`TotalPrice`, date components) and export the cleaned dataset and the returns dataset into `data/processed/`.

In [14]:
# Output directory (relative to the notebooks/ folder); create it if missing
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Feature Engineering: TotalPrice
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df_returns["TotalPrice"] = df_returns["Quantity"] * df_returns["UnitPrice"]

# 2. Feature Engineering: Date-time components
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

# 3. Export datasets to CSV (the "/" operator joins path parts)
cleaned_file_path = output_dir / "online_retail_cleaned.csv"
returns_file_path = output_dir / "online_retail_returns.csv"

df.to_csv(cleaned_file_path, index=False)
df_returns.to_csv(returns_file_path, index=False)

# Verification Step 6
total_rev = df["TotalPrice"].sum()
print("--- Verification Step 6 ---")
print("Cleaned dataset saved to: " + cleaned_file_path.as_posix())
print("Cleaned dataset row count: " + format(len(df), ","))
print("Cleaned dataset columns: " + str(list(df.columns)))
print("Total Sales Revenue: " + format(total_rev, ",.2f"))
print("Returns dataset saved to: " + returns_file_path.as_posix())
print("Returns dataset row count: " + format(len(df_returns), ","))

--- Verification Step 6 ---
Cleaned dataset saved to: ../data/processed/online_retail_cleaned.csv
Cleaned dataset row count: 522,537
Cleaned dataset columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TotalPrice', 'Year', 'Month', 'Day', 'DayOfWeek', 'Hour']
Total Sales Revenue: 10,247,219.32
Returns dataset saved to: ../data/processed/online_retail_returns.csv
Returns dataset row count: 9,725
